# DSC360 Milestone 4

**Author:** James Sneddon  
**Date:** May 18, 2026  
**Modified By:** James Sneddon  

## Description
Sentiment classification on IMDB 50K reviews. Builds on Milestone 3 feature engineering to compare multiple models and feature combinations.

In [1]:
import subprocess, sys, ssl

packages = ["nltk", "beautifulsoup4", "scikit-learn", "pandas", "numpy", "scipy"]
for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", pkg],
        stderr=subprocess.DEVNULL
    )

import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

for resource in ["stopwords", "wordnet", "punkt", "punkt_tab", "omw-1.4"]:
    nltk.download(resource, quiet=True)

import os
import re
import pickle
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from scipy.sparse import hstack
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

## 1. Load and Prepare Data

In [2]:
df = pd.read_csv('data/IMDB Dataset.csv')

print("Shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nClass balance:")
print(df["sentiment"].value_counts())

before = len(df)
df = df.drop_duplicates()
print(f"\nDropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

df["label"] = (df["sentiment"] == "positive").astype(int)

X = df["review"]
y = df["label"]

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nTrain: {len(X_train_raw)}, Val: {len(X_val_raw)}, Test: {len(X_test_raw)}")

Shape: (50000, 2)

First five rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Class balance:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Dropped 418 duplicate rows. Remaining: 49582

Train: 34707, Val: 7437, Test: 7438



Dropped 418 duplicate rows. Remaining: 49582

Train: 34707, Val: 7437, Test: 7438


## 2. Feature Combinations

In [3]:
def normalize_text(text: str) -> str:
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub("[^a-z]", " ", text)
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOP_WORDS]
    return " ".join(tokens)

_cache = 'data/imdb_normalized.pkl'
if os.path.exists(_cache):
    with open(_cache, 'rb') as f:
        train_clean, val_clean, test_clean = [pd.Series(x) for x in pickle.load(f)]
    print('Loaded from cache.')
else:
    train_clean = X_train_raw.reset_index(drop=True).apply(normalize_text)
    val_clean   = X_val_raw.reset_index(drop=True).apply(normalize_text)
    test_clean  = X_test_raw.reset_index(drop=True).apply(normalize_text)
    with open(_cache, 'wb') as f:
        pickle.dump([list(train_clean), list(val_clean), list(test_clean)], f)
    print('Done.')

print('Sample:', train_clean.iloc[0][:120])

Loaded from cache.
Sample: work library expected like movie came year ago well liked parker posey lot shes wonderful actress omar townsend really c


In [ ]:
tfidf_uni = TfidfVectorizer(max_features=5000)
A_train = tfidf_uni.fit_transform(train_clean)
A_val   = tfidf_uni.transform(val_clean)
A_test  = tfidf_uni.transform(test_clean)

tfidf_ng = TfidfVectorizer(ngram_range=(2, 3), max_features=3000)
B_train = hstack([A_train, tfidf_ng.fit_transform(train_clean)])
B_val   = hstack([A_val,   tfidf_ng.transform(val_clean)])
B_test  = hstack([A_test,  tfidf_ng.transform(test_clean)])

print("Feature A:", A_train.shape)
print("Feature B:", B_train.shape)

## 3. Logistic Regression

In [ ]:
lr_a = LogisticRegression(max_iter=1000, random_state=42)
lr_a.fit(A_train, y_train)
pred_a = lr_a.predict(A_val)

lr_b = LogisticRegression(max_iter=1000, random_state=42)
lr_b.fit(B_train, y_train)
pred_b = lr_b.predict(B_val)

print(f"Feature A  Accuracy: {accuracy_score(y_val, pred_a):.4f}  F1: {f1_score(y_val, pred_a):.4f}")
print(f"Feature B  Accuracy: {accuracy_score(y_val, pred_b):.4f}  F1: {f1_score(y_val, pred_b):.4f}")

Feature B did slightly better than Feature A. Adding bigrams and trigrams helped the model catch phrases like "not good" or "waste of time" that single words would miss, so the extra features were worth including. Feature B goes forward.

## 4. Nonlinear Classifier

In [ ]:
svc = LinearSVC(max_iter=2000, random_state=42)
svc.fit(B_train, y_train)
pred_svc = svc.predict(B_val)

print(f"Accuracy : {accuracy_score(y_val, pred_svc):.4f}")
print(f"F1 Score : {f1_score(y_val, pred_svc):.4f}")

LinearSVC came in just below Logistic Regression. The two models were close enough that the algorithm itself was not really the issue. LR on Feature B is still the best combination.

## 5. Final Evaluation

In [ ]:
pred_test = lr_b.predict(B_test)

print(f"Accuracy : {accuracy_score(y_test, pred_test):.4f}")
print(f"F1 Score : {f1_score(y_test, pred_test):.4f}")

Test results came in almost exactly the same as validation, which means the model holds up on data it has not seen before. The evaluation approach worked.

## 6. Observations

Feature B did better than Feature A across the board, which makes sense because adding short phrases on top of single words gives the model more to work with. Logistic Regression edged out LinearSVC by a small amount, though the two were close enough that either could win with different settings. The best combination was Logistic Regression on Feature B, coming in around 0.887 on both validation and test, and the scores matched closely enough that the model is clearly not just memorizing the training data. Worth trying next would be word embeddings to capture meaning rather than just exact word matches, or something like BERT that looks at full sentences instead of individual tokens.